In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 3.5 The Multipole Expansion

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume III — Classical Electrodynamics",
    number="3.5",
    title="The Multipole Expansion",
    blurb="What a charge distribution looks like from far away: the potential as "
    "a series in 1/r, the monopole–dipole–quadrupole moments, and the Legendre "
    "polynomials and spherical harmonics that organise the angular structure.",
    difficulty="advanced",
    estimate="120–150 min",
)

## Notebook overview

Stand far enough from any localized clump of charge and its details blur: a
complicated molecule, seen from across the room, is just a point with a slight
lopsidedness. The **multipole expansion** makes that intuition exact. It writes the
potential of a distribution as a series in powers of $1/r$, ordered so that each
term captures a coarser feature than the last: the **monopole** (the net charge,
falling as $1/r$), the **dipole** (the separation of $+$ from $-$, as $1/r^2$), the
**quadrupole** (the shape, as $1/r^3$), and on. Far away, the lowest non-zero
moment dominates, so a handful of numbers, the moments, are the distribution's
fingerprint seen from afar.

Organising that series brings in two families of **special functions**: the
**Legendre polynomials** $P_l$, which carry the angular structure of an
axially symmetric problem, and the **spherical harmonics** $Y_l^m$, which carry the
general angular structure on the sphere. We do not pull these from a hat. For each,
we state the differential equation it solves, write the first few members out in
closed form, state the **orthogonality** that makes it a usable basis (and say why
orthogonality is exactly what lets us *project* a distribution onto it and read off
the moments), point to where the full derivation lives, and then evaluate it with
SciPy while *checking SciPy against the closed forms*. That is the standard this
notebook sets for special functions: rigorous about what the tool is, without
rederiving Sturm–Liouville theory from scratch.

Two forward-looking notes. The $Y_l^m$ here are, exactly, the angular part of the
hydrogen-atom wavefunctions; the quantum-mechanics volume (Vol VI) inherits this
machinery and only adds the radial piece, so this is a down-payment on quantum
mechanics. And the very same expansion describes gravity: Earth's equatorial bulge
is a mass quadrupole, the $J_2$ term that makes satellite orbits precess.

Everything is in **SI units** ($\varepsilon_0 = 8.854\times10^{-12}\,$F/m,
$k=1/4\pi\varepsilon_0$). The objects here are static angular patterns and
converging series, so every figure is a still: there is no motion to animate.

> **How to read the checks.** Each exercise ends with a `validate` call against an
> independent fact: a generating function reproduced, an orthogonality integral
> equal to $2/(2n+1)$, a SciPy special function matched to its closed form, a
> truncated series converging to the exact potential. A ✓ is strong evidence; a ✗
> is a prompt to *locate the discrepancy*, not a verdict.

> **Scope.** A working review, not a special-functions course. See Nolting,
> *Theoretical Physics 3* {cite}`nolting3`; Griffiths, *Introduction to
> Electrodynamics* {cite}`griffiths_em` (ch. 3); Jackson {cite}`jackson` (ch. 3–4)
> for the full multipole and spherical-harmonic treatment; and Arfken, Weber &
> Harris {cite}`arfken` for the Sturm–Liouville origin of the special functions.

## Theory in brief

### The far-field problem

For a charge distribution $\rho(\mathbf r')$ confined to a small region near the
origin, the potential at a distant field point $\mathbf r$ is

```{math}
:label: eq-multipole-setup
\varphi(\mathbf r) = \frac{1}{4\pi\varepsilon_0}\int
\frac{\rho(\mathbf r')}{|\mathbf r - \mathbf r'|}\,d^3r' ,
```

and when $r \gg r'$ we can expand the kernel $1/|\mathbf r-\mathbf r'|$ in the small
ratio $r'/r$. The expansion is a systematic answer to "what does this distribution
look like from far away?"

### The Legendre expansion of the Coulomb kernel

With $\gamma$ the angle between $\mathbf r$ and $\mathbf r'$,
$|\mathbf r-\mathbf r'| = r\sqrt{1 - 2(r'/r)\cos\gamma + (r'/r)^2}$, and the kernel
expands as

```{math}
:label: eq-generating
\frac{1}{|\mathbf r-\mathbf r'|}
= \frac{1}{r}\sum_{l=0}^{\infty}\Big(\frac{r'}{r}\Big)^{l} P_l(\cos\gamma) ,
```

where the **Legendre polynomials** $P_l$ enter through their **generating function**
$1/\sqrt{1-2xt+t^2} = \sum_l P_l(x)\,t^l$. This is *where* the Legendre polynomials
come from in electrostatics; Griffiths {cite}`griffiths_em` (ch. 3) carries the
expansion out term by term. They solve **Legendre's equation**

$$ (1-x^2)\,y'' - 2x\,y' + l(l+1)\,y = 0, $$

and the first few are, in closed form,

$$ P_0 = 1,\quad P_1 = x,\quad P_2 = \tfrac12(3x^2-1),\quad P_3 = \tfrac12(5x^3-3x). $$

### Multipole moments

Substituting {eq}`eq-generating` into {eq}`eq-multipole-setup` groups the potential
into terms of definite falloff,

```{math}
:label: eq-moments
\varphi(\mathbf r) = \frac{1}{4\pi\varepsilon_0}\left[
\frac{Q}{r} + \frac{\mathbf p\cdot\hat{\mathbf r}}{r^2}
+ \frac{1}{2}\frac{\sum_{ij}Q_{ij}\hat r_i\hat r_j}{r^3} + \cdots \right],
```

with the **monopole** $Q=\int\rho\,d^3r'$ (total charge), the **dipole moment**
$\mathbf p=\int\mathbf r'\rho\,d^3r'$, and the **quadrupole tensor**
$Q_{ij}=\int(3r'_ir'_j - r'^2\delta_{ij})\rho\,d^3r'$. Each term falls one power of
$r$ faster than the last, so far away the lowest non-zero moment dominates.

### Orthogonality and projection

The moments are obtained by **projecting** the distribution onto an orthogonal
basis. That is the whole reason orthogonality matters: if a basis $\{f_n\}$
satisfies $\langle f_m, f_n\rangle = N_n\delta_{mn}$, then the coefficient of $f_n$
in any expansion is just $\langle f_n, \cdot\rangle/N_n$, read off with a single
integral. For Legendre polynomials (a Sturm–Liouville consequence; Arfken, Weber &
Harris {cite}`arfken` supply the proof),

```{math}
:label: eq-orthogonality
\int_{-1}^{1} P_m(x)\,P_n(x)\,dx = \frac{2}{2n+1}\,\delta_{mn} .
```

### Spherical harmonics

When the distribution is not axially symmetric, the angular dependence needs the
full **spherical harmonics** $Y_l^m(\theta,\varphi)$, the eigenfunctions of the
angular part of the Laplacian,

```{math}
:label: eq-spherical-harmonics
\frac{1}{\sin\theta}\partial_\theta(\sin\theta\,\partial_\theta Y_l^m)
+ \frac{1}{\sin^2\theta}\partial_\varphi^2 Y_l^m = -l(l+1)\,Y_l^m ,
```

the natural orthonormal basis on the sphere,
$\int Y_l^{m\,*}Y_{l'}^{m'}\,d\Omega = \delta_{ll'}\delta_{mm'}$. The first few are

$$ Y_0^0 = \tfrac{1}{\sqrt{4\pi}},\quad
Y_1^0 = \sqrt{\tfrac{3}{4\pi}}\cos\theta,\quad
Y_1^{\pm1} = \mp\sqrt{\tfrac{3}{8\pi}}\sin\theta\,e^{\pm i\varphi}. $$

Their derivation is separation of variables of Laplace's equation in spherical
coordinates, a Sturm–Liouville problem; we treat that as deferred (Arfken
{cite}`arfken`; a candidate dedicated notebook). These are exactly the angular
hydrogen states of Vol VI. And the same expansion, with mass for charge, gives the
**gravitational** multipoles: Earth's oblateness is a quadrupole, the $J_2$ term of
satellite geodesy.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import eval_legendre, sph_harm_y
from scipy.integrate import quad

from ecp import draw, validate

# NOTE on the SciPy API: scipy.special.sph_harm was REMOVED. We use
# sph_harm_y(n, m, theta, phi) with n the degree (l), m the order, theta the polar
# angle in [0, π], and phi the azimuth in [0, 2π]. Legendre polynomials come from
# eval_legendre(l, x). We verify both against their closed forms below.

from scipy.constants import epsilon_0 as EPS0  # vacuum permittivity, F/m

K = 1.0 / (4.0 * np.pi * EPS0)  # Coulomb constant ≈ 8.99e9 N·m²/C²
NANO = 1e-9  # 1 nC
POS, NEG = "#c1121f", "#16213e"  # positive red, negative dark blue


def point_potential(charges, P):
    """Exact superposed potential of a set of point charges.

    The direct Coulomb sum of k·q_i/|P − r_i|, the truth the multipole series
    is checked against.

    Parameters
    ----------
    charges : list of tuple
        Each ``(q, r_i)``: a charge and its 3-D position.
    P : numpy.ndarray
        Field points of shape ``(..., 3)``.

    Returns
    -------
    numpy.ndarray
        The exact potential at ``P``, in volts.
    """
    P = np.asarray(P, float)
    total = np.zeros(P.shape[:-1])
    for q, r0 in charges:
        total = total + K * q / np.linalg.norm(P - np.asarray(r0, float), axis=-1)
    return total

## Exercise 1 — Legendre polynomials from the generating function

The Legendre polynomials are not an arbitrary choice of basis: they fall out of the
Coulomb kernel itself, through the **generating function**
$1/\sqrt{1-2xt+t^2}=\sum_l P_l(x)\,t^l$ {eq}`eq-generating`. They solve Legendre's
equation $(1-x^2)y''-2xy'+l(l+1)y=0$, with closed forms $P_0=1$, $P_1=x$,
$P_2=\tfrac12(3x^2-1)$, $P_3=\tfrac12(5x^3-3x)$.

**This exercise.**

1. Verify the generating function numerically: at $x=\cos\gamma=0.37$ and
   $t=r'/r=0.6$, confirm the truncated sum
   $\sum_l$ `scipy.special.eval_legendre`$(l,x)\,t^l$ reproduces
   $1/\sqrt{1-2xt+t^2}$.
2. Check `eval_legendre` against the closed forms $P_0\ldots P_3$ on a grid
   $x\in[-1,1]$ ({numref}`fig-mp-legendre`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    gen_rhs, gen_lhs, "the generating function reproduces 1/√(1−2xt+t²)", rtol=1e-8
)
validate.close(
    max_closed_dev,
    0.0,
    "scipy's Legendre polynomials match the closed forms",
    atol=1e-12,
)

## Exercise 2 — Orthogonality and projection

What makes the Legendre polynomials a *usable* basis is their **orthogonality**
{eq}`eq-orthogonality`: $\int_{-1}^{1}P_m P_n\,dx = \tfrac{2}{2n+1}\delta_{mn}$.
Orthogonality is exactly what lets us **project**: the coefficient of $P_n$ in any
expansion $f=\sum_n c_n P_n$ is $c_n=\tfrac{2n+1}{2}\int_{-1}^{1}f\,P_n\,dx$, a
single integral. The multipole moments are this projection, applied to a charge
distribution.

**This exercise.**

1. Compute $\int_{-1}^{1}P_m P_n\,dx$ for $m,n\le3$ with
   `scipy.integrate.quad` and confirm the diagonal is $2/(2n+1)$ and the
   off-diagonal vanishes.
2. **Project** $f(x)=|x|$ onto the first twelve Legendre polynomials
   (coefficients $c_n$ by `scipy.integrate.quad`) and reconstruct it,
   watching the partial sums converge ({numref}`fig-mp-projection`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    diagonal,
    expected_diag,
    "the Legendre polynomials are orthogonal with norm 2/(2n+1)",
    rtol=1e-6,
)
validate.close(
    off_diag,
    0.0,
    "distinct Legendre polynomials are orthogonal (zero overlap)",
    atol=1e-9,
)

## Exercise 3 — The Legendre expansion of the Coulomb kernel

With the generating function in hand, the kernel expansion {eq}`eq-generating`
follows directly. For a source point on the $z$-axis at distance $d$ and a field
point at radius $r>d$ and polar angle $\theta$, the angle between them is $\theta$
itself, so $1/|\mathbf r-\mathbf r'| = (1/r)\sum_l (d/r)^l P_l(\cos\theta)$.

**This exercise.** Take $d=0.3\,$m on the $z$-axis and field points at
$r=1.0\,$m over $\theta\in[0,\pi]$. Confirm the truncated sum (24 terms of
`scipy.special.eval_legendre`) reproduces the exact kernel
$1/\sqrt{r^2+d^2-2rd\cos\theta}$.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    kernel_series,
    kernel_exact,
    "the Legendre expansion reproduces the Coulomb kernel 1/|r−r'|",
    rtol=1e-6,
)

## Exercise 4 — Multipole moments of a discrete distribution

For a set of point charges the integrals become sums: the **monopole** is
$Q=\sum_i q_i$ and the **dipole moment** is $\mathbf p=\sum_i q_i\mathbf r_i$
{eq}`eq-moments`. Truncating the expansion after the dipole gives the far-field
approximation $\varphi\approx (kQ)/r + k\,\mathbf p\cdot\hat{\mathbf r}/r^2$, which
should track the exact superposed potential ever better as $r$ grows
({numref}`fig-mp-discrete-setup`).

**This exercise.** Take $q_1=+1\,$nC at $(0,0,0.03)$, $q_2=-1\,$nC at
$(0.02,0,0)$, and $q_3=+0.5\,$nC at $(0,0,-0.02)\,$m.

1. Compute $Q=\sum_i q_i$ and $\mathbf p=\sum_i q_i\mathbf r_i$.
2. Evaluate the monopole-only and monopole+dipole approximations along an
   outward ray, and compare both to the exact superposed potential
   ({numref}`fig-mp-discrete-potential`). Adding the dipole term should
   sharply cut the far-field error.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    err_dip < err_mono / 10.0,
    "adding the dipole term sharply improves the far-field potential",
    f"monopole {err_mono:.3%} → monopole+dipole {err_dip:.2e}",
)

## Exercise 5 — The pure dipole and quadrupole

The two leading non-trivial moments have signature angular patterns
({numref}`fig-mp-dipole-quad-setup`). A **point dipole** (two opposite charges a
small distance apart) has potential $\varphi=k\,p\cos\theta/r^2$, the $P_1=\cos\theta$
pattern. A **linear quadrupole** ($+q$ at $+a\hat{\mathbf z}$, $-2q$ at the origin,
$+q$ at $-a\hat{\mathbf z}$) has zero charge and zero dipole, so its leading term is
the quadrupole, $\varphi\propto P_2(\cos\theta)/r^3$.

**This exercise.** Build both configurations ($q=1\,$nC, $a=0.01\,$m).

1. Confirm the exact dipole potential, scaled by $r^2/(kp)$, equals
   $P_1=\cos\theta$.
2. Confirm the exact quadrupole potential at large $r$, normalised, carries
   the $P_2(\cos\theta)$ pattern (`scipy.special.eval_legendre`)
   ({numref}`fig-mp-dipole-quad-angular`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    dipole_angular_err,
    0.0,
    "the dipole potential carries the P₁=cosθ angular pattern",
    atol=1e-3,
)
validate.close(
    quad_angular_err,
    0.0,
    "the linear-quadrupole far field carries the P₂(cosθ) pattern",
    atol=2e-2,
)

## Exercise 6 — Spherical harmonics

Drop axial symmetry and the angular structure needs the full **spherical harmonics**
$Y_l^m(\theta,\varphi)$ {eq}`eq-spherical-harmonics`, eigenfunctions of the angular
Laplacian with eigenvalue $-l(l+1)$, orthonormal on the sphere,
$\int Y_l^{m\,*}Y_{l'}^{m'}\,d\Omega=\delta_{ll'}\delta_{mm'}$. The first few are
$Y_0^0=1/\sqrt{4\pi}$, $Y_1^0=\sqrt{3/4\pi}\cos\theta$,
$Y_1^{\pm1}=\mp\sqrt{3/8\pi}\sin\theta\,e^{\pm i\varphi}$. Their derivation
(separation of Laplace's equation, a Sturm–Liouville problem) is deferred to
{cite}`arfken`.

**This exercise.** Using `scipy.special.sph_harm_y(l, m, theta, phi)` (the
current API; the old `sph_harm` was removed):

1. Confirm $\int|Y_l^m|^2\,d\Omega=1$ for several $(l,m)$ by integrating
   with `numpy.trapezoid` over $\theta,\varphi$ with the $\sin\theta$
   measure.
2. Verify $Y_1^0$ against its closed form $\sqrt{3/4\pi}\cos\theta$.
3. **Visualise the angular structure** in 3-D.

A
subtlety guides *what* to plot: $|Y_l^m|^2$ is **axially symmetric** for every
$(l,m)$, because the $\varphi$-dependence $e^{im\varphi}$ has unit modulus, so a
balloon of $|Y_l^m|^2$ is just the $\theta$-profile spun about the axis and shows
nothing a 2-D plot would not — the azimuthal structure lives in the **phase**.
Plot instead the **real** spherical harmonics $Y_{lm}^{\mathrm{real}}\propto
P_l^m(\cos\theta)\cos m\varphi$ (or $\sin|m|\varphi$ for $m<0$) as 3-D surfaces with
radius $\propto|Y_{lm}^{\mathrm{real}}|$ and colour set by its sign: these are the
familiar lobed orbital shapes, and the $m\neq0$ cases show the genuine azimuthal
lobes a 2-D polar plot cannot ({numref}`fig-mp-ylm`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    norm_err, 0.0, "the spherical harmonics are normalized on the sphere", atol=1e-2
)
validate.close(Y10_err, 0.0, "Y₁⁰ matches its closed form √(3/4π)cosθ", atol=1e-12)

## Exercise 7 — The general multipole expansion in spherical harmonics

For a distribution with no axial symmetry, the moments become the
**spherical-harmonic multipole moments**
$q_{lm}=\sum_i q_i\,r_i^{\,l}\,Y_l^{m\,*}(\theta_i,\varphi_i)$, and the exterior
potential is $\varphi(\mathbf r)=\tfrac{1}{\varepsilon_0}\sum_{l,m}
\tfrac{1}{2l+1}\,q_{lm}\,Y_l^m(\theta,\varphi)/r^{l+1}$
({numref}`fig-mp-offaxis-setup`).

**This exercise (student).** Take $q_1=+2\,$nC at $(0.02,0.01,0.0)$ and
$q_2=-2\,$nC at $(-0.01,0.02,0.015)\,$m (off every axis).

1. Compute the $q_{lm}$ for $l\le3$ with `scipy.special.sph_harm_y`.
2. Reconstruct the potential on a sphere of radius $0.4\,$m.
3. Confirm the relative error against the exact superposed potential
   **falls as more $(l,m)$ terms are kept**
   ({numref}`fig-mp-offaxis-error`).

These $Y_l^m$ are, identically, the angular hydrogen states of Vol VI: a
down-payment on quantum mechanics.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    monotone and errs[-1] < 1e-3,
    "the spherical-harmonic multipole series converges to the exact potential",
    f"errors by l: {[f'{e:.1e}' for e in errs]}",
)

## Exercise 8 — The gravitational quadrupole

The expansion is blind to whether its source is charge or mass: replace $q$ by $m$
(and $k$ by $-G$) and it describes **gravity**. A spherical body has only a
monopole, but a flattened one has a **quadrupole** ({numref}`fig-mp-oblate-setup`).
Earth is oblate, and its quadrupole is the famous $J_2$ term that makes satellite
orbits precess. The relevant moment is $Q_{zz}=\sum_i m_i(3z_i^2-r_i^2)$; for an
oblate body (mass pushed toward the equator, where $z^2\ll r^2$) it is **negative**,
and $J_2=-Q_{zz}/(2Ma^2)>0$.

**This exercise (student).** Model an oblate spheroid (equatorial radius
$a=1$, polar radius $c=0.9$, in scaled units) by sampling it with equal point
masses (a seeded `numpy.random.default_rng` rejection sample).

1. Compute $Q_{zz}=\sum_i m_i(3z_i^2-r_i^2)$.
2. Confirm it is negative and matches the uniform-spheroid closed form
   $\tfrac25 M(c^2-a^2)<0$, so the gravitational quadrupole (the $J_2$
   term) is non-zero with the sign oblateness demands.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.check(
    Q_zz < 0.0,
    "an oblate mass has a negative Q_zz, i.e. a positive gravitational quadrupole J₂",
    f"Q_zz = {Q_zz:.4f}",
)
validate.close(
    Q_zz,
    Q_zz_closed,
    "the sampled quadrupole matches the uniform-spheroid closed form (2/5)M(c²−a²)",
    rtol=2e-2,
)

## Exercise 9 — Why the multipole expansion matters

The thread running through every exercise is one idea: **the lowest non-zero moment
dominates the far field**, so a distribution we cannot resolve in detail is
characterised, from afar, by a few numbers. A neutral molecule is invisible as a
monopole but speaks through its dipole and quadrupole (the basis of intermolecular
forces in chemistry); the cosmic microwave background is described by its angular
multipoles $C_l$ (the cosmologist's spherical-harmonic spectrum of the sky); an
antenna's reach is set by its radiation multipoles
([§3.10](radiation.ipynb)). The moments are
the **fingerprint** of a source seen from a distance.

**This exercise.** Make the dominance concrete for the Exercise 4 cloud: along the
outward ray, evaluate the fraction of the exact potential captured by the monopole
alone, and watch it approach $1$ as $r$ grows, the higher moments fading as powers
of $1/r$ ({numref}`fig-mp-dominance`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 9

In [ ]:
validate.check(
    far_fraction > 0.9,
    "far away the monopole term captures almost all of the potential (lowest moment dominates)",
    f"monopole fraction at r=3 m = {far_fraction:.3f}",
)

## Notebook summary

- The Legendre polynomials from the generating function (`scipy.special.eval_legendre`,
  matched to the closed forms), their orthogonality $\int P_mP_n=2/(2n+1)\,\delta_{mn}$, and
  the Legendre expansion of the Coulomb kernel.
- Discrete monopole and dipole moments and the far-field truncation; the dipole ($P_1$) and
  quadrupole ($P_2$) angular patterns; the spherical harmonics via the current
  `scipy.special.sph_harm_y` API (rendered in 3-D as the real-harmonic orbital lobes,
  orthonormal to $\sim10^{-2}$); and the general $q_{lm}$ expansion converging to the exact
  potential.
- Earth's gravitational quadrupole ($J_2$) of an oblate spheroid, the same machinery applied
  to gravity. Established the tools-with-a-note standard for special functions.

## Outlook

- **The quadrupole tensor.** Its trace-free form and principal axes are the same
  mathematics as the rigid-body inertia tensor
  ([§2.6](../02-classical-mechanics/rigid-body.ipynb)); diagonalizing it gives the
  distribution's natural axes.
- **The full $Y_l^m$ for $m\ne0$.** Built from the associated Legendre functions
  $P_l^m$ (`scipy.special.assoc_legendre_p`/`lpmv`); the addition theorem for
  spherical harmonics ties the $Y_l^m$ basis back to the single $P_l(\cos\gamma)$ of
  {eq}`eq-generating`.
- **Forward links.** Radiation multipoles ([§3.10](radiation.ipynb)); the angular hydrogen states and
  the full Sturm–Liouville derivation of these special functions (Vol VI); the CMB
  and gravitational ($J_2$, and beyond) multipoles.

## References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()